# Exercice n°7 : Machine Learning pour la Neuroimagerie - Prédire le TDAH

Bienvenue dans ce deuxième exercice pratique. Vous allez construire un pipeline complet d'apprentissage automatique pour prédire le diagnostic de **Trouble du Déficit de l'Attention avec ou sans Hyperactivité (TDAH)** à partir de données de connectivité fonctionnelle IRM.

Vous utiliserez le jeu de données **ADHD200**, disponible via nilearn, qui contient des données d'IRMf de repos pour des sujets avec et sans diagnostic de TDAH.

Cet exercice vous fera appliquer les mêmes étapes que celles vues en classe :
1. Exploration des données phénotypiques
2. Extraction de features de connectivité fonctionnelle
3. Entraînement et évaluation d'un classifieur SVM

**Consignes importantes pour l'évaluation automatique :**

1. **Conformité** : Ne modifiez pas les noms des variables de résultat (ex: `q1_n_sujets`, `defi_accuracy`) donnés dans les lignes commentées. Ces noms sont essentiels pour l'évaluation automatique.
2. **Exécution** : Assurez-vous que votre code s'exécute sans erreur de haut en bas.
3. **Paramètres** : Respectez exactement les paramètres spécifiés dans chaque question (atlas, random_state, etc.). L'évaluateur automatique compare vos résultats à des valeurs de référence calculées avec ces paramètres précis.
4. **Données** : Utilisez uniquement les variables `pheno`, `func` et `confounds` fournies dans la cellule de configuration (Section 0), qui sont déjà alignées entre elles.

Bon courage !

## Section 0 : Configuration et chargement des données

Exécutez ces cellules de configuration. Elles téléchargent les données et préparent les variables que vous utiliserez dans l'exercice. **Ne les modifiez pas.**

In [ ]:
# Configuration — ne pas modifier
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nilearn import datasets, plotting
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Bibliothèques importées avec succès.")

In [ ]:
# Chargement du dataset ADHD200 — ne pas modifier
data_dir = './nilearn_data'

adhd_dataset = datasets.fetch_adhd(n_subjects=40, data_dir=data_dir)

# Les données phénotypiques brutes (peuvent ne pas couvrir tous les sujets)
# Note: On convertit en DataFrame pour assurer la compatibilité avec les méthodes pandas
pheno_brut = pd.DataFrame(adhd_dataset.phenotypic)

print(f"Dataset chargé : {len(adhd_dataset.func)} images fonctionnelles")
print(f"Données phénotypiques brutes : {pheno_brut.shape[0]} sujets")

In [ ]:
# Alignement des données — ne pas modifier
#
# Certains sujets ont des images IRMf mais pas de données phénotypiques.
# On conserve uniquement les sujets présents dans les deux sources.

func_ids = [int(os.path.basename(f).split('_')[0]) for f in adhd_dataset.func]
pheno_ids = set(pheno_brut['Subject'].values)
matched_idx = [i for i, fid in enumerate(func_ids) if fid in pheno_ids]
matched_fids = [func_ids[i] for i in matched_idx]

# Données alignées et prêtes à l'emploi
pheno    = pheno_brut.set_index('Subject').loc[matched_fids].reset_index()
func     = [adhd_dataset.func[i] for i in matched_idx]
confounds = [adhd_dataset.confounds[i] for i in matched_idx]

print(f"Sujets avec données d'imagerie ET phénotypiques : {len(func)}")
print(f"Colonnes phénotypiques disponibles : {list(pheno.columns)}")

---
## Partie 1 : Exploration des données phénotypiques (25 points)

Dans cette section, vous allez explorer les variables cliniques et démographiques du jeu de données ADHD200.

*Il y a 5 questions et chacune vaut 5 points.*

### Question 1 : Nombre de sujets

En utilisant la variable `pheno` fournie dans la Section 0, déterminez le nombre total de sujets disposant à la fois de données d'imagerie et de données phénotypiques.

Stockez ce nombre entier dans **`q1_n_sujets`**.

In [ ]:
# Réponse 1

# Votre code ici

# q1_n_sujets = ...

### Question 2 : Distribution des diagnostics

La colonne `adhd` du DataFrame `pheno` contient le diagnostic : `1` pour TDAH, `0` pour contrôle.

Comptez le nombre de sujets TDAH et le nombre de sujets contrôles.

Stockez les résultats dans **`q2_n_tdah`** (entier) et **`q2_n_controles`** (entier).

In [ ]:
# Réponse 2

# Votre code ici

# q2_n_tdah = ...
# q2_n_controles = ...

### Question 3 : Variable cible

Créez le vecteur cible `y` pour l'apprentissage automatique : un tableau NumPy d'entiers (dtype `int`) contenant `1` pour les sujets TDAH et `0` pour les contrôles, dans l'ordre de la variable `pheno`.

Stockez le résultat dans **`q3_y`**.

*Indice : utilisez la colonne `adhd` de `pheno` et convertissez en tableau NumPy d'entiers.*

In [ ]:
# Réponse 3

# Votre code ici

# q3_y = ...

### Question 4 : Âge moyen par groupe

Calculez l'âge moyen (colonne `age`) séparément pour les sujets TDAH et pour les contrôles.

Stockez l'âge moyen des sujets **TDAH** dans **`q4_age_moyen_tdah`** (float) et l'âge moyen des **contrôles** dans **`q4_age_moyen_controles`** (float).

Créez également un graphique (barplot ou boxplot) comparant la distribution d'âge entre les deux groupes.

In [ ]:
# Réponse 4

# Votre code ici

# q4_age_moyen_tdah = ...
# q4_age_moyen_controles = ...

### Question 5 : Recherche — Répartition des sexes

Consultez la doc : [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

La colonne `sex` contient `'M'` pour masculin et `'F'` pour féminin.

En utilisant `groupby`, calculez le nombre de sujets masculins (`'M'`) dans chaque groupe diagnostique (TDAH vs contrôle).

Stockez le nombre d'hommes parmi les sujets **TDAH** dans **`q5_n_hommes_tdah`** (entier) et parmi les **contrôles** dans **`q5_n_hommes_controles`** (entier).

*Indice : filtrez d'abord sur `sex == 'M'`, puis comptez par groupe `adhd`.*

In [ ]:
# Réponse 5

# Votre code ici

# q5_n_hommes_tdah = ...
# q5_n_hommes_controles = ...

---
## Partie 2 : Extraction de features de connectivité fonctionnelle (20 points)

Dans cette section, vous allez extraire les matrices de connectivité fonctionnelle pour chaque sujet en utilisant un atlas cérébral. Ces matrices constitueront les **features** (variables prédictives) de votre modèle ML.

Vous utiliserez :
- L'atlas **BASC multiscale** à résolution **12 ROIs**
- Le `NiftiLabelsMasker` de nilearn pour extraire les séries temporelles
- Le `ConnectivityMeasure` de nilearn pour calculer les corrélations

*Il y a 4 questions et chacune vaut 5 points.*

**Note :** L'extraction peut prendre quelques minutes. Une fois terminée, les features seront sauvegardées sur disque pour éviter de recalculer.

In [ ]:
# Chargement de l'atlas BASC 12 ROIs — ne pas modifier
basc = datasets.fetch_atlas_basc_multiscale_2015(
    data_dir=data_dir, resolution=12
)
atlas_img = basc.maps
print(f"Atlas chargé : {atlas_img}")

### Question 6 : Extraction des séries temporelles

Initialisez un `NiftiLabelsMasker` avec les paramètres suivants (obligatoires pour la validation automatique) :
- `labels_img=atlas_img`
- `standardize='zscore_sample'`
- `detrend=True`
- `resampling_target='data'`
- `verbose=0`

Utilisez ce masker pour extraire les séries temporelles du **premier sujet** (`func[0]`, `confounds[0]`). Stockez les séries temporelles dans **`q6_timeseries`**.

Ensuite, stockez le **nombre de régions d'intérêt (ROIs)** dans **`q6_n_rois`** (entier). C'est la deuxième dimension de `q6_timeseries`.

In [ ]:
# Réponse 6

# Votre code ici

# q6_timeseries = ...
# q6_n_rois = ...

### Question 7 : Matrice de connectivité vectorisée

En utilisant `ConnectivityMeasure` avec les paramètres suivants :
- `kind='correlation'`
- `vectorize=True`
- `discard_diagonal=True`

Calculez le vecteur de connectivité pour le premier sujet à partir de `q6_timeseries`.

Stockez le vecteur dans **`q7_corr_vecteur`** et le **nombre de features** (longueur du vecteur) dans **`q7_n_features`** (entier).

*Indice : `ConnectivityMeasure.fit_transform` attend une liste de tableaux de séries temporelles.*

In [ ]:
# Réponse 7

# Votre code ici

# q7_corr_vecteur = ...
# q7_n_features = ...

### Question 8 : Extraction pour tous les sujets

Répétez l'extraction pour **tous les sujets** de la liste `func` (en utilisant les `confounds` correspondants). Construisez la matrice de features `X` en empilant les vecteurs de connectivité.

Stockez la matrice résultante dans **`q8_X_features`** (tableau NumPy 2D).

Pour économiser du temps lors des re-exécutions, le code de sauvegarde/chargement est fourni dans la cellule suivante.

*Indice : utilisez une boucle `for` et la même approche que pour le sujet 0.*

In [ ]:
# Réponse 8
# Initialisez un nouveau ConnectivityMeasure avec les mêmes paramètres qu'à la Question 7

# Votre code ici — construisez q8_X_features avec une boucle sur tous les sujets

# q8_X_features = ...

### Question 9 : Vérification de la forme de la matrice

Vérifiez que la matrice `q8_X_features` a la bonne forme : elle doit avoir autant de lignes que de sujets, et autant de colonnes que de features de connectivité.

Stockez la **forme (shape)** de la matrice dans **`q9_X_shape`** (tuple).

Visualisez également la matrice avec `plt.imshow` pour inspecter visuellement les features (axes : sujets en ordonnée, features en abscisse).

In [ ]:
# Réponse 9

# Votre code ici

# q9_X_shape = ...

---
## Partie 3 : Pipeline d'apprentissage automatique (25 points)

Vous allez maintenant entraîner un classifieur SVM sur les features de connectivité extraites pour prédire le diagnostic TDAH.

Utilisez **`q8_X_features`** comme matrice de features et **`q3_y`** comme vecteur cible.

*Il y a 5 questions et chacune vaut 5 points.*

### Question 10 : Séparation entraînement / test

Divisez les données en un ensemble d'entraînement et un ensemble de test en utilisant `train_test_split` avec les paramètres **obligatoires** suivants :
- `test_size=0.2`
- `shuffle=True`
- `stratify=q3_y` (pour maintenir la proportion TDAH/contrôle dans les deux ensembles)
- `random_state=0`

Stockez le **nombre de sujets d'entraînement** dans **`q10_n_train`** (entier) et le **nombre de sujets de test** dans **`q10_n_test`** (entier).

Visualisez la distribution TDAH/contrôle dans chaque ensemble pour vérifier l'équilibre.

In [ ]:
# Réponse 10

# Votre code ici
# X_train, X_test, y_train, y_test = train_test_split(...)

# q10_n_train = ...
# q10_n_test = ...

### Question 11 : Standardisation des données

Appliquez un `StandardScaler` aux données :
1. Ajustez (*fit*) le scaler **uniquement** sur `X_train`
2. Transformez `X_train` → `X_train_scl`
3. Transformez `X_test` → `X_test_scl` avec **le même scaler** (sans le ré-ajuster)

Stockez la **moyenne globale** de `X_train_scl` dans **`q11_moyenne_train_scl`** (float).

*Rappel : après standardisation sur les données d'entraînement, la moyenne de `X_train_scl` doit être très proche de zéro. Pourquoi ne faut-il pas ré-ajuster le scaler sur `X_test` ?*

In [ ]:
# Réponse 11

# Votre code ici

# q11_moyenne_train_scl = ...

### Question 12 : Validation croisée sur l'ensemble d'entraînement

Initialisez un classifieur SVM avec les paramètres **obligatoires** suivants :
- `kernel='linear'`
- `class_weight='balanced'`

Obtenez les prédictions par validation croisée à **3 plis** (`cv=3`) sur l'ensemble d'entraînement.

Stockez les **prédictions de la validation croisée** dans **`q12_y_pred_cv`** (tableau NumPy).

*Indice : `cross_val_predict(estimateur, X, y, cv=3)`*

In [ ]:
# Réponse 12

# Votre code ici

# q12_y_pred_cv = ...

### Question 13 : Accuracy de la validation croisée

Calculez l'**accuracy globale** du modèle sur l'ensemble d'entraînement en comparant `q12_y_pred_cv` et `y_train`.

Stockez l'accuracy dans **`q13_cv_accuracy`** (float entre 0 et 1).

Affichez également le rapport de classification complet avec `classification_report`.

*Réflexion : le modèle performe-t-il mieux que le hasard (50%) ? Qu'est-ce qui explique une performance modeste avec seulement 24 sujets d'entraînement et 66 features ?*

In [ ]:
# Réponse 13

# Votre code ici

# q13_cv_accuracy = ...

### Question 14 : Matrice de confusion

Calculez la **matrice de confusion** à partir de `y_train` et `q12_y_pred_cv`.

Stockez la matrice dans **`q14_cm`** (tableau NumPy 2×2).

Visualisez-la avec `sns.heatmap` (avec les labels `['Contrôle', 'TDAH']` pour les axes).

*Rappel : la diagonale représente les prédictions correctes. Quel type d'erreur (faux positif ou faux négatif) est le plus fréquent ?*

In [ ]:
# Réponse 14

# Votre code ici

# q14_cm = ...

---
## Défi : Évaluation finale sur données non vues (30 points)

C'est le moment de vérité ! Vous allez entraîner le modèle sur **toutes** les données d'entraînement et l'évaluer sur l'ensemble de test que vous avez mis de côté depuis le début.

Cette étape simule l'utilisation réelle d'un modèle en clinique : le modèle ne voit les données de test **qu'une seule fois**, après que toutes les décisions de conception ont été prises.

Suivez ces étapes :
1. Entraînez le classifieur SVM (mêmes paramètres qu'à la Q12) sur `X_train_scl` et `y_train`
2. Prédisez les étiquettes pour `X_test_scl`
3. Calculez l'accuracy sur le test set

Stockez les **prédictions finales** dans **`defi_y_pred`** (tableau NumPy) et l'**accuracy finale** dans **`defi_accuracy`** (float).

*Réflexion : comparez l'accuracy du test set à celle de la validation croisée. Sont-elles similaires ? Que vous dit cette comparaison sur la généralisation du modèle ? Avec seulement 6 sujets dans le test set, quelle est la fiabilité de cette estimation ?*

In [ ]:
# Défi

# Votre code ici
# Étape 1 : entraîner le modèle final sur X_train_scl, y_train
# Étape 2 : prédire sur X_test_scl
# Étape 3 : calculer l'accuracy

# defi_y_pred = ...
# defi_accuracy = ...

---
## Pour aller plus loin (non noté)

Si vous avez terminé, explorez ces pistes :

1. **Autre atlas** : refaites l'extraction avec `resolution=36` (630 features). Le modèle est-il meilleur ou pire ? Pourquoi ?
2. **Test de permutation** : utilisez `permutation_test_score` pour vérifier si la performance du modèle est statistiquement significative malgré le petit nombre de sujets.
3. **Visualisation des poids** : entraînez le modèle final, récupérez `svc.coef_` et utilisez `correlation_measure.inverse_transform` pour afficher les connexions les plus importantes sous forme de connectome.
4. **Régression** : remplacez le diagnostic binaire par une mesure continue (ex: score symptomatique `dsm_iv_tot`) et utilisez un `SVR` (Support Vector Regressor) au lieu d'un `SVC`.